In [ ]:
import pandas as pd

files = {
    "Food Prices": "../data/food_prices.csv",
    "Rainfall":    "../data/rainfall.csv",
    "Nutrition":   "../data/nutrition.csv",
    "Crop Old":    "../data/crop_old.csv",
    "Crop New":    "../data/crop_new.csv",
}

for name, path in files.items():
    try:
        df = pd.read_csv(path)
        print(f"\n{'='*40}")
        print(f"✅ {name}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"❌ {name} — ERROR: {e}")


✅ Food Prices
Shape: (2117473, 6)
Columns: ['id', 'date', 'state_name', 'state_code', 'commodity', 'price']
   id        date      state_name  state_code commodity  price
0   0  2015-01-01  Andhra Pradesh          28      Rice   26.0
1   1  2015-01-01           Assam          18      Rice   24.0

✅ Rainfall
Shape: (204876, 8)
Columns: ['id', 'date', 'state_code', 'state_name', 'actual', 'rfs', 'normal', 'deviation']
   id        date  state_code   state_name  actual       rfs  normal  \
0   0  2009-01-01           5  Uttarakhand     0.0  0.003906    2.19   
1   1  2009-01-01          18        Assam     0.0  0.000000    0.52   

   deviation  
0     -100.0  
1     -100.0  

✅ Nutrition
Shape: (110, 137)
Columns: ['States/UTs', 'Area', 'Number of Households surveyed', 'Number of Women age 15-49 years interviewed', 'Number of Men age 15-54 years interviewed', 'Female population age 6 years and above who ever attended school (%)', 'Population below age 15 years (%)', ' Sex ratio of the t

In [6]:
import pandas as pd
import numpy as np
import os

os.makedirs("../data/cleaned", exist_ok=True)
print("Folders ready")

Folders ready


In [23]:
# ── Re-clean food prices with correct commodity names ────
food_raw = pd.read_csv("../data/food_prices.csv")

# Correct target commodities
target = ['Rice', 'Wheat', 'Tur/Arhar Dal', 'Soya Oil (Packed)']
food_raw = food_raw[food_raw['commodity'].isin(target)]

# Parse dates
food_raw['date'] = pd.to_datetime(food_raw['date'])

# Pivot to wide
food_wide = food_raw.pivot_table(
    index=['state_name', 'date'],
    columns='commodity',
    values='price',
    aggfunc='mean'
).reset_index()

food_wide.columns.name = None
food_wide = food_wide.rename(columns={
    'state_name': 'state',
    'Rice': 'rice_price',
    'Wheat': 'wheat_price',
    'Tur/Arhar Dal': 'dal_price',
    'Soya Oil (Packed)': 'oil_price'
})

# Convert to monthly
food_wide['date'] = food_wide['date'].dt.to_period('M').dt.to_timestamp()
food_monthly = food_wide.groupby(
    ['state', 'date'], as_index=False
).mean(numeric_only=True)

food_monthly.to_csv("../data/cleaned/food_prices_cleaned.csv", index=False)
print(f"Food prices fixed: {food_monthly.shape}")
print(food_monthly.head(3))
print(f"\nNull counts:\n{food_monthly.isnull().sum()}")

Food prices fixed: (3741, 6)
                         state       date  rice_price  oil_price   dal_price  \
0  Andaman And Nicobar Islands 2015-01-01       40.03        NaN  114.849091   
1  Andaman And Nicobar Islands 2015-02-01       36.00        NaN   90.750000   
2  Andaman And Nicobar Islands 2015-03-01       36.00        NaN   96.470588   

   wheat_price  
0       34.375  
1       27.000  
2       27.000  

Null counts:
state            0
date             0
rice_price       0
oil_price      416
dal_price       85
wheat_price    354
dtype: int64


In [11]:
rain = pd.read_csv("../data/rainfall.csv")

rain = rain[['date','state_name','actual']].copy()
rain = rain.rename(columns={
    'state_name': 'State',
    'actual': 'rainfall_mm'
})

rain['date'] = pd.to_datetime(rain['date'])
rain['rainfall_mm'] = pd.to_numeric(rain['rainfall_mm'], errors='coerce')

rain.to_csv("../data/cleaned/rainfall_cleaned.csv", index=False)
print(f"Rainfall cleaned : {rain.shape}")
print(rain.head(3))

Rainfall cleaned : (204876, 3)
        date        State  rainfall_mm
0 2009-01-01  Uttarakhand          0.0
1 2009-01-01        Assam          0.0
2 2009-01-01      Tripura          0.0


In [26]:
nutrition = pd.read_csv("../data/nutrition.csv")    

nutrition = nutrition[[
    'States/UTs',
    'Children under 5 years who are stunted (height-for-age)18 (%)',
    'Children under 5 years who are underweight (weight-for-age)18 (%)',
    'Children age 6-59 months who are anaemic (<11.0 g/dl)22 (%)'
]].copy()

nutrition.columns = ['state', 'stunting_rate', 'underweight_rate', 'anaemia_rate']

nutrition = nutrition[~nutrition['state'].str.contains(
    'India|Total|Rural|Urban', case=False, na=False
)]

for col in ['stunting_rate', 'underweight_rate', 'anaemia_rate']:
    nutrition[col] = nutrition[col].astype(str)
    nutrition[col] = nutrition[col].str.replace(r'[^0-9.]', '', regex=True)
    nutrition[col] = pd.to_numeric(nutrition[col], errors='coerce')

nutrition = nutrition.dropna(subset=['state'])
nutrition = nutrition.reset_index(drop=True)   

nutrition.to_csv("../data/cleaned/nutrition_cleaned.csv", index=False)
print(f"Nutrition cleaned : {nutrition.shape}")
print(nutrition.head(5))

Nutrition cleaned : (107, 4)
                       state  stunting_rate  underweight_rate  anaemia_rate
0  Andaman & Nicobar Islands           18.2              15.1          47.8
1  Andaman & Nicobar Islands           26.4              31.1          33.3
2  Andaman & Nicobar Islands           22.5              23.7          40.0
3             Andhra Pradesh           23.1              25.1          58.7
4             Andhra Pradesh           34.2              31.4          65.0


In [18]:
# ── Old crop file (1997-2020) ────────────────────────────
crop_old = pd.read_csv("../data/crop_old.csv")
crop_old.columns = [c.strip().lower() for c in crop_old.columns]

# Target crops only
target_crops = ['Rice', 'Wheat', 'Maize', 'Pulses']
crop_old = crop_old[crop_old['crop'].isin(target_crops)]

# Aggregate district → state level
crop_old = crop_old.groupby(
    ['state', 'crop', 'crop_year'], as_index=False
).agg(
    area=('area', 'sum'),
    production=('production', 'sum'),
    yield_val=('yield', 'mean')
)
crop_old = crop_old.rename(columns={
    'crop_year': 'year',
    'yield_val': 'yield'
})

# ── New crop file (2021-2023) ────────────────────────────
crop_new = pd.read_csv("../data/crop_new.csv")
crop_new.columns = ['sl_no', 'state', 'category', '2021', '2022', '2023']

# Extract crop name
crop_new['crop'] = crop_new['category'].str.replace(
    'Productivity of ', '', regex=False
).str.strip()

# Keep target crops only
crop_new = crop_new[crop_new['crop'].isin(target_crops)]

# Melt to long format
crop_new = crop_new.melt(
    id_vars=['state', 'crop'],
    value_vars=['2021', '2022', '2023'],
    var_name='year',
    value_name='yield'
)
crop_new['year'] = crop_new['year'].astype(int)
crop_new['area'] = np.nan
crop_new['production'] = np.nan

# ── Combine both ─────────────────────────────────────────
crop_combined = pd.concat([crop_old, crop_new], ignore_index=True)
crop_combined = crop_combined.sort_values(['state', 'crop', 'year'])

# ── Create state-year summary for master merge ───────────
crop_summary = crop_combined.groupby(
    ['state', 'year'], as_index=False
).agg(
    yield_ton_per_hectare=('yield', 'mean'),
    production_tonnes=('production', 'sum')
)

crop_summary.to_csv("../data/cleaned/crop_yield_cleaned.csv", index=False)
print(f"Crop yield cleaned: {crop_summary.shape}")
print(f"Year range: {crop_summary['year'].min()} to {crop_summary['year'].max()}")
print(crop_summary.head(3))

Crop yield cleaned: (834, 4)
Year range: 1997 to 2023
                        state  year  yield_ton_per_hectare  production_tonnes
0  Andaman and Nicobar Island  2000                  3.055            32184.0
1  Andaman and Nicobar Island  2001                  3.195            27333.0
2  Andaman and Nicobar Island  2002                  2.825            32112.0


In [19]:
import os

files = [
    "../data/cleaned/food_prices_cleaned.csv",
    "../data/cleaned/rainfall_cleaned.csv",
    "../data/cleaned/nutrition_cleaned.csv",
    "../data/cleaned/crop_yield_cleaned.csv",
]

for f in files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        print(f"✅ {os.path.basename(f)} — {df.shape}")
    else:
        print(f"❌ Missing: {f}")

✅ food_prices_cleaned.csv — (96184, 5)
✅ rainfall_cleaned.csv — (204876, 3)
✅ nutrition_cleaned.csv — (107, 4)
✅ crop_yield_cleaned.csv — (834, 4)


In [22]:
# ── Fix 1: Food prices → monthly averages ───────────────
food = pd.read_csv("../data/cleaned/food_prices_cleaned.csv")
food['date'] = pd.to_datetime(food['date'])
food['date'] = food['date'].dt.to_period('M').dt.to_timestamp()
food = food.groupby(['state', 'date'], as_index=False).mean(numeric_only=True)
print(f"Food prices after monthly aggregation: {food.shape}")
print(food.head(3))

# ── Check oil commodity name ─────────────────────────────
raw_food = pd.read_csv("../data/food_prices.csv")
print("\nAll unique commodities in your file:")
print(raw_food['commodity'].unique())

Food prices after monthly aggregation: (3741, 5)
                         state       date  rice_price   dal_price  wheat_price
0  Andaman And Nicobar Islands 2015-01-01       40.03  114.849091       34.375
1  Andaman And Nicobar Islands 2015-02-01       36.00   90.750000       27.000
2  Andaman And Nicobar Islands 2015-03-01       36.00   96.470588       27.000

All unique commodities in your file:
<StringArray>
[                  'Rice',                  'Wheat',           'Atta (Wheat)',
               'Gram Dal',          'Tur/Arhar Dal',               'Urad Dal',
              'Moong Dal',             'Masoor Dal',                  'Sugar',
                   'Milk', 'Groundnut Oil (Packed)',   'Mustard Oil (Packed)',
     'Vanaspati (Packed)',      'Soya Oil (Packed)', 'Sunflower Oil (Packed)',
      'Palm Oil (Packed)',                    'Gur',              'Tea Loose',
    'Salt Pack (Iodised)',                 'Potato',                  'Onion',
                 'Tomato']
Len

In [27]:
# ── Fix nutrition duplicate state rows ───────────────────
nutrition = pd.read_csv("../data/cleaned/nutrition_cleaned.csv")

# Keep only Total rows — drop rural/urban
# Each state should appear only once
nutrition = nutrition.drop_duplicates(subset='state', keep='first')

# Drop India aggregate
nutrition = nutrition[~nutrition['state'].str.contains(
    'India|Total', case=False, na=False
)]

nutrition = nutrition.reset_index(drop=True)
nutrition.to_csv("../data/cleaned/nutrition_cleaned.csv", index=False)
print(f"Nutrition fixed: {nutrition.shape}")
print(nutrition.head(5))

Nutrition fixed: (36, 4)
                       state  stunting_rate  underweight_rate  anaemia_rate
0  Andaman & Nicobar Islands           18.2              15.1          47.8
1             Andhra Pradesh           23.1              25.1          58.7
2          Arunachal Pradesh           28.4              13.1          52.8
3                      Assam           29.8              25.9          66.4
4                      Bihar           36.8              35.8          67.9


In [30]:
# ── Load all cleaned files ───────────────────────────────
food = pd.read_csv("../data/cleaned/food_prices_cleaned.csv")
rain = pd.read_csv("../data/cleaned/rainfall_cleaned.csv")
nutrition = pd.read_csv("../data/cleaned/nutrition_cleaned.csv")
crop = pd.read_csv("../data/cleaned/crop_yield_cleaned.csv")

# ── Fix date formats ─────────────────────────────────────
food['date'] = pd.to_datetime(food['date'])
rain['date'] = pd.to_datetime(rain['date'])

# ── Standardize state column names ──────────────────────
rain = rain.rename(columns={'State': 'state'})

# ── Add year column ──────────────────────────────────────
food['year'] = food['date'].dt.year

# ── Drop year from rain before merge to avoid conflict ───
rain = rain.drop(columns=['year'], errors='ignore')

# ── Merge food + rainfall on state + date ────────────────
master = pd.merge(food, rain, on=['state', 'date'], how='left')

# ── Merge crop on state + year ───────────────────────────
master = pd.merge(master, crop, on=['state', 'year'], how='left')

# ── Merge nutrition on state only ───────────────────────
master = pd.merge(master, nutrition, on='state', how='left')

# ── Save ─────────────────────────────────────────────────
master.to_csv("../data/master.csv", index=False)

print(f"✅ Master CSV shape: {master.shape}")
print(f"Columns: {master.columns.tolist()}")
print(f"\nNull counts:\n{master.isnull().sum()}")
print(master.head(3))

✅ Master CSV shape: (3741, 13)
Columns: ['state', 'date', 'rice_price', 'oil_price', 'dal_price', 'wheat_price', 'year', 'rainfall_mm', 'yield_ton_per_hectare', 'production_tonnes', 'stunting_rate', 'underweight_rate', 'anaemia_rate']

Null counts:
state                       0
date                        0
rice_price                  0
oil_price                 416
dal_price                  85
wheat_price               354
year                        0
rainfall_mm               278
yield_ton_per_hectare    1381
production_tonnes        1381
stunting_rate             489
underweight_rate          489
anaemia_rate              489
dtype: int64
                         state       date  rice_price  oil_price   dal_price  \
0  Andaman And Nicobar Islands 2015-01-01       40.03        NaN  114.849091   
1  Andaman And Nicobar Islands 2015-02-01       36.00        NaN   90.750000   
2  Andaman And Nicobar Islands 2015-03-01       36.00        NaN   96.470588   

   wheat_price  year  rainf

In [31]:
# Check unique state names in each file
states_food = set(food['state'].unique())
states_rain = set(rain['state'].unique())
states_nutrition = set(nutrition['state'].unique())
states_crop = set(crop['state'].unique())

print("In food but NOT in rainfall:")
print(states_food - states_rain)

print("\nIn food but NOT in nutrition:")
print(states_food - states_nutrition)

print("\nIn food but NOT in crop:")
print(states_food - states_crop)

In food but NOT in rainfall:
set()

In food but NOT in nutrition:
{'The Dadra And Nagar Haveli And Daman And Diu', 'Maharashtra', 'Jammu And Kashmir', 'Andaman And Nicobar Islands', 'Delhi'}

In food but NOT in crop:
{'Jammu And Kashmir', 'The Dadra And Nagar Haveli And Daman And Diu', 'Andaman And Nicobar Islands', 'Chandigarh'}


In [32]:
# ── State name standardization map ──────────────────────
state_map = {
    # Nutrition fixes
    'Andaman & Nicobar Islands': 'Andaman And Nicobar Islands',
    'Jammu & Kashmir': 'Jammu And Kashmir',
    'Dadra & Nagar Haveli and Daman & Diu': 'The Dadra And Nagar Haveli And Daman And Diu',
    'NCT of Delhi': 'Delhi',
    'Delhi': 'Delhi',

    # Crop fixes
    'Andaman and Nicobar Island': 'Andaman And Nicobar Islands',
    'Jammu and Kashmir': 'Jammu And Kashmir',
    'Dadra and Nagar Haveli': 'The Dadra And Nagar Haveli And Daman And Diu',
}

# Apply to nutrition and crop
nutrition['state'] = nutrition['state'].replace(state_map)
crop['state'] = crop['state'].replace(state_map)

# Verify fixes
print("Remaining in food but NOT in nutrition:")
print(set(food['state'].unique()) - set(nutrition['state'].unique()))

print("\nRemaining in food but NOT in crop:")
print(set(food['state'].unique()) - set(crop['state'].unique()))

Remaining in food but NOT in nutrition:
{'The Dadra And Nagar Haveli And Daman And Diu', 'Maharashtra'}

Remaining in food but NOT in crop:
{'Chandigarh'}


In [33]:
# ── Rebuild master with fixed state names ────────────────
master = pd.merge(food, rain, on=['state', 'date'], how='left')
master = pd.merge(master, crop, on=['state', 'year'], how='left')
master = pd.merge(master, nutrition, on='state', how='left')

master.to_csv("../data/master.csv", index=False)

print(f"✅ Master CSV shape: {master.shape}")
print(f"\nNull counts:\n{master.isnull().sum()}")

✅ Master CSV shape: (3741, 13)

Null counts:
state                       0
date                        0
rice_price                  0
oil_price                 416
dal_price                  85
wheat_price               354
year                        0
rainfall_mm               278
yield_ton_per_hectare    1225
production_tonnes        1225
stunting_rate             145
underweight_rate          145
anaemia_rate              145
dtype: int64


In [34]:
# Check what Maharashtra looks like in nutrition file
nutrition_raw = pd.read_csv("../data/cleaned/nutrition_cleaned.csv")
maha = nutrition_raw[nutrition_raw['state'].str.contains('Maharashtra', case=False, na=False)]
print("Maharashtra in nutrition:")
print(maha)

# Check exact state name
print("\nAll nutrition states:")
print(sorted(nutrition['state'].unique()))

Maharashtra in nutrition:
Empty DataFrame
Columns: [state, stunting_rate, underweight_rate, anaemia_rate]
Index: []

All nutrition states:
['Andaman And Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli & Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharastra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [35]:
# Which states have null stunting_rate
null_states = master[master['stunting_rate'].isna()]['state'].unique()
print("States with null nutrition data:")
print(sorted(null_states))

States with null nutrition data:
['Maharashtra', 'The Dadra And Nagar Haveli And Daman And Diu']


In [36]:
# Fix typos in nutrition
nutrition['state'] = nutrition['state'].replace({
    'Maharastra': 'Maharashtra',
    'Dadra and Nagar Haveli & Daman and Diu': 'The Dadra And Nagar Haveli And Daman And Diu',
})

# Rebuild master
master = pd.merge(food, rain, on=['state', 'date'], how='left')
master = pd.merge(master, crop, on=['state', 'year'], how='left')
master = pd.merge(master, nutrition, on='state', how='left')

master.to_csv("../data/master.csv", index=False)

print(f"✅ Master CSV shape: {master.shape}")
print(f"\nNull counts:\n{master.isnull().sum()}")

✅ Master CSV shape: (3741, 13)

Null counts:
state                       0
date                        0
rice_price                  0
oil_price                 416
dal_price                  85
wheat_price               354
year                        0
rainfall_mm               278
yield_ton_per_hectare    1225
production_tonnes        1225
stunting_rate               0
underweight_rate            0
anaemia_rate                0
dtype: int64
